# TraceCore Record Mode Colab Demo

This notebook is optimized for Google Colab. It walks through the *make-or-break* record flow:

1. Install TraceCore from PyPI.
2. Resolve a packaged reference agent + deterministic task.
3. Run `agent-bench run --record` to seal (or discard) a baseline bundle.
4. Inspect the resulting artifacts and verify bundle integrity.

Record mode replays the agent immediately after sealing the bundle. If the replay drifts (different actions, outputs, or budgets), the bundle is deleted—hence *make-or-break*.

## 1. Install TraceCore
Colab VMs start clean, so install the published package each session.

In [ ]:
%pip install -q tracecore

print("✅ tracecore installed")


## 2. Locate a packaged agent + task
TraceCore ships reference agents. We'll use the `log_alert_triage` pairing which is deterministic and quick (<30s).

In [ ]:
import os

from importlib import resources

from pathlib import Path


agent_path = resources.files("agent_bench.agents").joinpath("log_alert_triage_agent.py")

task_ref = "log_alert_triage@1"

print(f"Agent path: {agent_path}")

print(f"Task ref: {task_ref}")

os.environ["TRACECORE_AGENT"] = str(agent_path)

os.environ["TRACECORE_TASK"] = task_ref

## 3. Run make-or-break record mode
This command will:
- Run the agent once and seal a baseline bundle.
- Immediately re-run with the same seed.
- Delete the bundle if the replay drifts.

> **Tip:** Record mode writes artifacts under `.agent_bench/` in the working directory.

In [ ]:
%%bash

set -euo pipefail

agent-bench run \

  --agent "$TRACECORE_AGENT" \

  --task "$TRACECORE_TASK" \

  --seed 0 \

  --record

## 4. Inspect the sealed artifacts
The most recent run + bundle live under `.agent_bench/runs` and `.agent_bench/baselines`.

In [ ]:
from pathlib import Path

runs = sorted(Path('.agent_bench/runs').glob('*.json'), key=lambda p: p.stat().st_mtime, reverse=True)

if not runs:

    raise SystemExit('No runs found; re-run record cell')

latest_run = runs[0]

print(f'Latest run artifact: {latest_run}')

print(latest_run.read_text()[:400] + '...')

baseline_dir = Path('.agent_bench/baselines') / latest_run.stem

print(f'Baseline bundle location: {baseline_dir}')

## 5. Verify bundle integrity
Use the same CLI verifier that CI uses. It checks hash manifests and sandbox allowlists.

In [ ]:
import subprocess, json, textwrap

result = subprocess.run(
    ['agent-bench', 'baseline', '--verify', str(baseline_dir)],

    check=True, capture_output=True, text=True

)

print(result.stdout)

## Next steps
- Commit the recorded bundle (if this is your canonical baseline).
- Push it through CI record/replay templates (GitHub Actions or GitLab).
- Use `agent-bench export otlp` + the Colab environment to push traces into your preferred backend.